# Phase 2: Machine Learning Baseline & Model Selection

**Source (PDF Section: WHAT YOU NEED TO BUILD - 2):** "Train a model... the goal is speed and cost. A trained classifier running in 2ms for free is not the same product as an expensive LLM call."

In this notebook, we will:
1. **Vectorize** our text and brand data.
2. **Train** four different classification models.
3. **Evaluate** them based on Accuracy, F1-Score, and Latency (Inference Speed).
4. **Export** the winner for production use in our FastAPI backend.

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, accuracy_score

# Paths
DATA_PATH = "../data/"
MODELS_PATH = "../backend/models/" # We save directly to the backend folder for easy access later

# Ensure the models folder exists
os.makedirs(MODELS_PATH, exist_ok=True)

# Load our processed data
train_df = pd.read_csv(f"{DATA_PATH}train.csv")
val_df = pd.read_csv(f"{DATA_PATH}val.csv")

print(f"Loaded {len(train_df)} training samples and {len(val_df)} validation samples.")

Loaded 71616 training samples and 23872 validation samples.


## Step 2: Feature Engineering
We use a `ColumnTransformer` to handle our two different types of data:
* **Text Data:** We use TF-IDF (Term Frequency-Inverse Document Frequency) to convert words into numerical importance scores.
* **Categorical Data:** We use One-Hot Encoding for the `author_id` so the model knows which brand handled the tweet.

In [4]:
# Define the features and target
X_train = train_df[['clean_text', 'author_id']]
y_train = train_df['priority']

X_val = val_df[['clean_text', 'author_id']]
y_val = val_df['priority']

# Updated Preprocessor with 2D column selection
preprocessor = ColumnTransformer(
    transformers=[
        # Tfidf can handle a Series, but 'brand' strictly needs a 2D slice
        ('text', TfidfVectorizer(max_features=5000, stop_words='english'), 'clean_text'),
        ('brand', OneHotEncoder(handle_unknown='ignore'), ['author_id']) # Note the brackets!
    ]
)

print("Preprocessing pipeline defined.")

Preprocessing pipeline defined.


## Step 3: The Model Tournament (Comparison & Selection)
**Source (PDF Section: WHAT YOU NEED TO BUILD - 2):** "The real question... is when is a cheap specialized model worth more than an expensive general one?"

In this step, we evaluate four different "classical" machine learning architectures to find our baseline. We aren't just looking for the highest accuracy; we are looking for the best **Decision Intelligence** balance between:
1. **F1-Score:** Because our data is unbalanced (73/27), accuracy can be misleading. F1-Score ensures the model is actually good at catching the "High Priority" cases.
2. **Inference Latency (ms):** To meet the 2ms production requirement, the model must be lightweight.
3. **Training Time:** As we scale to 3M+ rows, we need to know if the model remains practical to retrain.

### The Contestants:
* **Logistic Regression:** The "Speed King" for text classification.
* **Naive Bayes:** Highly efficient with TF-IDF features.
* **Linear SVC:** Often superior for high-dimensional text data.
* **Random Forest:** A robust ensemble method to capture non-linear relationships.

In [5]:
def evaluate_model(name, model, X_t, y_t, X_v, y_v):
    """
    Trains a model and returns its performance metrics and average latency.
    """
    # Create a full pipeline with our preprocessor and the specific model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Measure Training Time
    start_train = time.time()
    pipeline.fit(X_t, y_t)
    train_time = time.time() - start_train
    
    # Measure Latency (Inference Time)
    # We measure the time taken to predict on the entire validation set and divide
    start_infer = time.time()
    preds = pipeline.predict(X_v)
    latency = (time.time() - start_infer) / len(X_v) * 1000 # convert to ms per tweet
    
    # Calculate Metrics
    f1 = f1_score(y_v, preds)
    acc = accuracy_score(y_v, preds)
    
    return {
        'Model': name,
        'Accuracy': acc,
        'F1-Score': f1,
        'Latency (ms)': latency,
        'Train Time (s)': train_time,
        'Pipeline': pipeline
    }

# Define our 4 contestants
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Naive Bayes': MultinomialNB(),
    'Linear SVC': LinearSVC(class_weight='balanced', dual=False),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1)
}

# Run the Tournament
results = []
for name, model in models.items():
    print(f"Training {name}...")
    results.append(evaluate_model(name, model, X_train, y_train, X_val, y_val))

# Display the Results
results_df = pd.DataFrame(results).drop(columns='Pipeline')
print("\n--- Tournament Results ---")
print(results_df.sort_values(by='F1-Score', ascending=False))

Training Logistic Regression...
Training Naive Bayes...
Training Linear SVC...
Training Random Forest...

--- Tournament Results ---
                 Model  Accuracy  F1-Score  Latency (ms)  Train Time (s)
2           Linear SVC  0.997528  0.995395      0.006126        0.607818
0  Logistic Regression  0.994135  0.989006      0.006469        0.610420
1          Naive Bayes  0.831434  0.720167      0.006323        0.464834
3        Random Forest  0.861009  0.652638      0.007518        0.791897


Step 4: Saving the "Champion"
We need to save the winning pipeline and the tournament results so your FastAPI backend and React frontend can use them later.

In [6]:
# 1. Identify the best model pipeline from the tournament
# Since Linear SVC was the winner in your run:
best_pipeline = [res['Pipeline'] for res in results if res['Model'] == 'Linear SVC'][0]

# 2. Save the full pipeline (Preprocessor + Model)
# This ensures the backend knows how to clean/vectorize new input
joblib.dump(best_pipeline, f"{MODELS_PATH}priority_model.joblib")

# 3. Save the metrics as a CSV for the Frontend Dashboard
# We remove the 'Pipeline' object before saving to CSV
results_df.to_csv(f"{MODELS_PATH}model_metrics.csv", index=False)

print(f"Champion model saved to: {MODELS_PATH}priority_model.joblib")
print(f"Metrics saved to: {MODELS_PATH}model_metrics.csv")

Champion model saved to: ../backend/models/priority_model.joblib
Metrics saved to: ../backend/models/model_metrics.csv


Step 5: The Final Test (The "Gold Standard")
The PDF says every number must come from code you actually ran. To be 100% honest, we need to run our winner against the Test Set (the 20% we locked away in Phase 1) just once.

In [7]:
# Load the Test Set
test_df = pd.read_csv(f"{DATA_PATH}test.csv")
X_test = test_df[['clean_text', 'author_id']]
y_test = test_df['priority']

# Final Prediction
test_preds = best_pipeline.predict(X_test)

print("--- FINAL TEST SET PERFORMANCE (Linear SVC) ---")
print(classification_report(y_test, test_preds))

--- FINAL TEST SET PERFORMANCE (Linear SVC) ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17437
           1       1.00      0.99      1.00      6436

    accuracy                           1.00     23873
   macro avg       1.00      1.00      1.00     23873
weighted avg       1.00      1.00      1.00     23873

